# All the ways to `eval`/`eval_and_grad`

`Propagator` supports three interchangeable **backends** for evaluating
observables and their gradients (`"standard"`, `"sparse"`, `"vmap"`), plus an
orthogonal **threading** knob (`eval_n_jobs`). All combinations compute
*exactly* the same values and gradients (verified below, and in
[`tests/test_backends.py`](../tests/test_backends.py)).

This notebook goes further than a bare backend comparison - it answers four
concrete questions, each with its own dedicated section:

1. **Why does `eval_n_jobs` bring no speedup, even with 128 CPU cores
   available?** (Step 6) - measured directly, with a sweep over
   `eval_n_jobs ∈ {1, 8, 16, 32, 64, -1}` for both `"standard"` and
   `"sparse"`, plus a control experiment that isolates pure thread-dispatch
   overhead from the actual NumPy work.
2. **Is pprop's `"sparse"` backend an actual sparse matrix?** No - it's a
   *narrow dense* array (`build_sparse_arrays`/`make_sparse_evaluator` in
   `pprop/propagator/utils.py`). Step 7 builds a genuine `scipy.sparse`
   variant (real CSR matrices for the gradient scatter-add) to see whether
   literal sparse linear algebra does any better.
3. **Does JAX support sparse arrays?** Yes, via `jax.experimental.sparse`
   (Step 8) - and Step 9 uses it to attack the one JAX-specific bottleneck
   already identified in
   [`tests/test_eval_and_grad_jit_bench.py`](../tests/test_eval_and_grad_jit_bench.py):
   the gradient's final scatter-add under `jax.vmap` on CPU. It genuinely
   helps *within* the `"vmap"` family once wired correctly (a real
   compile-time trap had to be found and fixed along the way - see Step 9)
   - though not enough to beat `"sparse"`.
4. **For every implementation above, how is the propagated observable
   actually stored, and what does "evaluate" mean for that representation?**
   Each section prints the real shapes/dtypes involved, not just a prose
   description.

Every section that produces a timing number appends it to one Python list
(`ALL_RESULTS`); Step 10 prints all of them together as a single
plain-Python table - deliberately *not* a markdown table, so the ranking is
reproduced by re-running the notebook rather than hand-copied.

Circuit, observables, `SIDE`, and `NUM_OBS` (Step 1) are unchanged throughout
- every backend and every variant below is evaluated against the exact same
propagated observables, so the comparisons are apples-to-apples. Because of
that, and because this notebook now also sweeps `eval_n_jobs` and tries two
extra sparse-matrix variants, the whole notebook takes several minutes to
run end to end (dominated by Step 2's three propagations, ~90s each).

**Bottom line up front** (measured on this machine, 128 CPU cores available,
CPU-only JAX - see Step 10 for the full table): `backend="sparse"` at
`eval_n_jobs=1` is still the recommended configuration - nothing tried here
beats it. Thread-pool `eval_n_jobs` (any width) and a `scipy.sparse`-backed
scatter were both neutral-to-worse. A properly-wired `jax.experimental.sparse`
scatter measurably speeds up the `"vmap"` family itself (Step 9) - so JAX
sparse arrays are not a dead end in general - but `"vmap"` still trails
`"sparse"` even with that fix, so the recommendation doesn't change.


## Step 0: Imports

In [1]:
import os
import time
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pennylane as qml
import scipy.sparse as sp

from pprop import Propagator
from pprop.propagator.pruning import DeadQubitPruner, XYWeightPruner
from pprop.propagator.utils import build_arrays, build_sparse_arrays

# Every section that measures a ms/call number appends one dict here; Step 10
# prints the whole thing as a single plain-Python table.
ALL_RESULTS: list[dict] = []


def record(implementation: str, eval_n_jobs, ms_per_call: float, notes: str = "") -> None:
    """Append one timing measurement to ALL_RESULTS (consumed by Step 10)."""
    ALL_RESULTS.append({
        "implementation": implementation,
        "eval_n_jobs": eval_n_jobs,
        "ms_per_call": ms_per_call,
        "notes": notes,
    })


/home/samonaco/Pauli-Propagator/.venv/lib/python3.12/site-packages/pennylane/operation.py:2622: PennyLaneDeprecationWarning: Observable is deprecated and will be removed in v0.43. A generic Operator class should be used instead. If defining an Operator, set the is_hermitian property to True. If checking if an Operator is Hermitian, check the is_hermitian property. 
  warnings.warn(


## Step 1: Circuit and observable set

`SIDE`, `NUM_OBS`, the circuit, and `OBS` are the ones this notebook's
findings are pinned to - unchanged throughout, including in every new
section below, so every backend/variant comparison is against the exact
same propagated observables.


In [2]:
SIDE, NUM_OBS, K1, K2, SEED, SIGMA = 8, 1000, 8, 32, 0, 3
num_qubits = SIDE * SIDE
rng = np.random.default_rng(SEED)


def sample_observables(
    rng: np.random.Generator,
) -> list[tuple[tuple[int, ...], object, int]]:
    t = np.tanh(1.0 / (4.0 * SIGMA ** 2))
    p = t / (1.0 + t)

    counts: dict[tuple[int, ...], int] = {}
    while len(counts) < NUM_OBS:
        mask   = rng.random(num_qubits) < p
        qubits = tuple(int(q) for q in np.where(mask)[0])
        if len(qubits) == 0:
            continue
        counts[qubits] = counts.get(qubits, 0) + 1

    pool = []
    for qubits, count in counts.items():
        ob = qml.PauliZ(qubits[0])
        for q in qubits[1:]:
            ob = ob @ qml.PauliZ(q)
        pool.append(ob)
    return pool

OBS = sample_observables(rng)

def circuit(params):
    index = 0

    # Initial RY and RX
    for q in range(SIDE*SIDE):
        qml.RY(params[index], wires=q)
        index += 1
        qml.RX(params[index], wires=q)
        index += 1

    # Horizontal entanglers
    for d in range(2):
        y_start = 0 if d % 2 == 0 else 1
        for x in range(SIDE):
            for y in range(y_start, SIDE - 1, 2):
                i = x * SIDE + y
                j = x * SIDE + (y + 1)
                qml.CNOT(wires=[i, j])

    # RX layer
    for q in range(SIDE*SIDE):
        qml.RX(params[index], wires=q)
        index += 1

    # Vertical entanglers
    for d in range(2):
        x_start = 0 if d % 2 == 0 else 1
        for y in range(SIDE):
            for x in range(x_start, SIDE - 1, 2):
                i = x * SIDE + y
                j = (x + 1) * SIDE + y
                qml.CNOT(wires=[i, j])

    # RX layer
    for q in range(SIDE*SIDE):
        qml.RX(params[index], wires=q)
        index += 1

    # Final RY
    for q in range(SIDE*SIDE):
        qml.RY(params[index], wires=q)
        index += 1
        
    return [qml.expval(ob) for ob in OBS]

## Step 2: The three backends

Build one `Propagator` per backend from the *same* circuit, so we can compare
them directly afterwards. `num_jobs=1` (propagation-time multiprocessing) is
used for all three here deliberately - see the warning below.

> **Why `num_jobs=1` in this notebook:** `backend="vmap"` uses JAX, which
> starts background threads as soon as it actually runs a computation.
> `propagate(num_jobs=...)` with `num_jobs > 1` uses `multiprocessing`'s
> `fork` start method - forking *after* JAX's threads exist can deadlock the
> child (it can inherit a mutex a thread mid-fork no longer has). This is a
> real, reproducible issue (see `Propagator.propagate`'s docstring), not
> theoretical. At this notebook's scale, `num_jobs=1` still finishes each
> propagation in well under two minutes (see the timing printed below); at
> larger scale, either keep `num_jobs=1` whenever `backend="vmap"` is
> anywhere in the same process, or avoid `backend="vmap"` (the recommended
> path anyway - see Step 10).


In [3]:
propagators = {}
for backend in ("standard", "sparse", "vmap"):
    t0 = time.perf_counter()
    prop = Propagator(circuit, k1=K1, k2=K2)
    prop.propagate(pruners=[XYWeightPruner(), DeadQubitPruner()], num_jobs=1, backend=backend)
    propagators[backend] = prop
    print(f"backend={backend:10s} propagate() took {time.perf_counter() - t0:6.2f}s "
          f"-> {len(prop.exprs)} observables, num_params={prop.num_params}")


backend=standard   propagate() took  88.90s -> 1000 observables, num_params=320
backend=sparse     propagate() took  88.81s -> 1000 observables, num_params=320
backend=vmap       propagate() took  90.90s -> 1000 observables, num_params=320


### Storage & evaluation, concretely

The intro cell describes these three backends in prose; here's what each one
actually holds in memory for this circuit's real propagated observables (not
a toy example) - one representative observable, plus aggregate stats across
all of them.

- **`"standard"`** (`build_arrays`/`make_evaluator`): every term gets a full
  `(num_params,)` row of integer sin/cos *powers* - `0` for every untouched
  parameter. Evaluation broadcasts `sin(theta)`/`cos(theta)` (each
  `(num_params,)`) against these `(n_terms, num_params)` count arrays with
  `**`, so a term that only touches a handful of parameters still pays for
  every untouched one via a no-op `x**0`.
- **`"sparse"`** (`build_sparse_arrays`/`make_sparse_evaluator`): only the
  parameters a term actually touches are stored, as an `(n_terms, W)` pair of
  `(index, power)` arrays, `W` = the widest term in that observable.
  Evaluation *gathers* `sin(theta)[idx_sin]`/`cos(theta)[idx_cos]` instead of
  broadcasting over the full width, so the array is exactly as wide as the
  work that needs doing. The gradient's per-term contributions land back in
  a full `(num_params,)` vector via `np.add.at` - a genuine scatter-add,
  since multiple terms can touch the same parameter.
- **`"vmap"`** (`build_sparse_arrays` again + `jax.vmap`/`jax.jit`): the same
  narrow per-observable arrays as `"sparse"`, but padded to a common
  `(n_obs, max_terms, W)` shape and stacked, so one compiled call replaces
  the Python loop over observables entirely. The forward pass is a batched
  version of `"sparse"`'s gather; the gradient's scatter-add becomes
  `.at[idx].add(...)`, JAX's traced equivalent of `np.add.at` - see Step 9
  for why that specific step is this backend's bottleneck.

Two more variants show up later, built directly in this notebook rather than
in `pprop` - a `scipy.sparse`-backed scatter (Step 7) and a
`jax.experimental.sparse`-backed one (Step 9) - each gets the same
storage/evaluation treatment in its own section.


In [4]:
num_params = propagators["standard"].num_params
exprs = propagators["standard"].exprs  # identical regardless of backend - only the
                                        # evaluator differs, not the propagated expression

sample_idx = int(np.argmax([len(e) for e in exprs]))  # widest observable, most informative
expr_sample = exprs[sample_idx]

coeffs_d, sin_counts_d, cos_counts_d = build_arrays(expr_sample, num_params)
coeffs_s, idx_sin_s, pow_sin_s, idx_cos_s, pow_cos_s = build_sparse_arrays(expr_sample, num_params)

n_terms = len(expr_sample)
print(f"Observable #{sample_idx}: {n_terms} terms, num_params={num_params}")
print(f"  standard : sin_counts{sin_counts_d.shape} cos_counts{cos_counts_d.shape}  dtype={sin_counts_d.dtype}"
      f"  ({sin_counts_d.size + cos_counts_d.size:,} entries, mostly zero)")
print(f"  sparse   : idx_sin{idx_sin_s.shape} pow_sin{pow_sin_s.shape}  idx_cos{idx_cos_s.shape} pow_cos{pow_cos_s.shape}"
      f"  ({idx_sin_s.size + idx_cos_s.size + pow_sin_s.size + pow_cos_s.size:,} entries)"
      f"  -> sin width W={idx_sin_s.shape[1]} vs num_params={num_params}"
      f" ({idx_sin_s.shape[1] / num_params:.1%} as wide)")

# Aggregate across every observable, not just the widest one.
widths_sin = [build_sparse_arrays(e, num_params)[1].shape[1] for e in exprs]
print(f"  sparse width W (sin) across all {len(widths_sin)} observables: "
      f"min={min(widths_sin)} mean={np.mean(widths_sin):.1f} max={max(widths_sin)}"
      f"  (num_params={num_params})")


Observable #994: 453 terms, num_params=320
  standard : sin_counts(453, 320) cos_counts(453, 320)  dtype=int32  (289,920 entries, mostly zero)
  sparse   : idx_sin(453, 12) pow_sin(453, 12)  idx_cos(453, 24) pow_cos(453, 24)  (32,616 entries)  -> sin width W=12 vs num_params=320 (3.8% as wide)
  sparse width W (sin) across all 1000 observables: min=1 mean=4.9 max=16  (num_params=320)


## Step 3: Calling `eval` and `eval_and_grad`

Same call, regardless of backend - `Propagator.__call__` and
`Propagator.eval_and_grad` dispatch internally based on what `propagate()`
was told to build.


In [5]:
params0 = rng.normal(size=propagators["standard"].num_params)

N_REPEAT_BASELINE = 10
for backend, prop in propagators.items():
    vals = prop(params0)                       # __call__: values only
    vals2, grads = prop.eval_and_grad(params0)  # values AND gradients together
    print(f"[{backend:10s}] eval(params).shape={vals.shape}  "
          f"eval_and_grad -> vals.shape={vals2.shape}, grads.shape={grads.shape}")

    _ = prop.eval_and_grad(params0)  # warm-up (JIT trace, for "vmap")
    t0 = time.perf_counter()
    for _ in range(N_REPEAT_BASELINE):
        prop.eval_and_grad(params0)
    ms = (time.perf_counter() - t0) / N_REPEAT_BASELINE * 1000
    print(f"              eval_and_grad: {ms:.3f} ms/call (eval_n_jobs=1)")
    if backend == "vmap":
        # standard/sparse get their eval_n_jobs=1 entry from Step 6's sweep
        # instead (same computation, avoids a duplicate row in Step 10).
        record("vmap", None, ms, notes="single-threaded baseline")


[standard  ] eval(params).shape=(1000,)  eval_and_grad -> vals.shape=(1000,), grads.shape=(1000, 320)
              eval_and_grad: 204.576 ms/call (eval_n_jobs=1)
[sparse    ] eval(params).shape=(1000,)  eval_and_grad -> vals.shape=(1000,), grads.shape=(1000, 320)
              eval_and_grad: 83.312 ms/call (eval_n_jobs=1)
[vmap      ] eval(params).shape=(1000,)  eval_and_grad -> vals.shape=(1000,), grads.shape=(1000, 320)
              eval_and_grad: 20.638 ms/call (eval_n_jobs=1)


## Step 4: Verifying all three agree

Not "close" in a loose sense - `"standard"` and `"sparse"` should match to
machine precision (they're the same math, just a narrower array), and
`"vmap"` should match closely too (small floating-point differences are
possible from JAX's different summation order).


In [6]:
reference_vals, reference_grads = propagators["standard"].eval_and_grad(params0)

for backend in ("sparse", "vmap"):
    vals, grads = propagators[backend].eval_and_grad(params0)
    val_diff = np.max(np.abs(vals - reference_vals))
    grad_diff = np.max(np.abs(grads - reference_grads))
    print(f"[{backend:10s}] max |value diff| = {val_diff:.2e}   max |grad diff| = {grad_diff:.2e}")
    assert np.allclose(vals, reference_vals, atol=1e-8)
    assert np.allclose(grads, reference_grads, atol=1e-8)

print("\nAll backends agree.")


[sparse    ] max |value diff| = 1.11e-16   max |grad diff| = 3.89e-16
[vmap      ] max |value diff| = 1.11e-16   max |grad diff| = 2.22e-16

All backends agree.


## Step 5: `eval_n_jobs` - threading across observables

Independent of `backend` (for `"standard"`/`"sparse"`): threads the Python
loop over observables using NumPy's GIL release during array ops. `-1` uses
every CPU actually allocated to this process (`os.sched_getaffinity`, so it
respects a SLURM/cgroup allocation rather than the whole machine) - on this
machine that resolves to all 128 cores.

First, a quick correctness check at `eval_n_jobs=-1`. Step 6 below is the
actual investigation into *why*, despite 128 cores being available, this
knob measures no advantage (and usually a regression) at any thread count.


In [7]:
prop_threaded = Propagator(circuit, k1=K1, k2=K2)
prop_threaded.propagate(
    pruners=[XYWeightPruner(), DeadQubitPruner()],
    num_jobs=1,
    backend="sparse",
    eval_n_jobs=-1,
)
print(f"eval_n_jobs resolved to: {prop_threaded.eval_n_jobs}  "
      f"(os.sched_getaffinity reports {len(os.sched_getaffinity(0))} CPUs available)")

vals_threaded, grads_threaded = prop_threaded.eval_and_grad(params0)
assert np.allclose(vals_threaded, reference_vals, atol=1e-10)
assert np.allclose(grads_threaded, reference_grads, atol=1e-10)
print("Threaded sparse backend matches the single-threaded reference exactly.")


eval_n_jobs resolved to: 128  (os.sched_getaffinity reports 128 CPUs available)
Threaded sparse backend matches the single-threaded reference exactly.


## Step 6: Why doesn't `eval_n_jobs` help, even with 128 cores?

Two things need to both be true for thread-pool parallelism to pay off:
(a) each task must release the GIL for long enough that CPython's GIL
acquire/release and thread-scheduling overhead is small *relative* to that
task, and (b) there must be idle cores to soak up. `(b)` is satisfied here -
`os.sched_getaffinity` reports 128 cores below. This section checks `(a)`
directly: how long does a *single* observable's `eval_and_grad` call
actually take?

Then two experiments, both sweeping `eval_n_jobs` over
`{1, 8, 16, 32, 64, -1}`:

1. **The real workload** - for `backend="standard"` and `backend="sparse"`,
   reusing the exact `_eval_and_grad_list` closures already built in Step 2
   (no re-propagation needed - `eval_n_jobs` only changes how the existing
   per-observable closures are dispatched, not what they compute).
2. **A control: pure dispatch overhead** - the same thread-pool machinery
   dispatching `len(observables)` no-op tasks instead of real work, to
   isolate "cost of asking N threads to do something" from "cost of the
   NumPy work itself".


In [8]:
JOB_COUNTS = [1, 8, 16, 32, 64, -1]

n_cpus = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else os.cpu_count()
print(f"os.sched_getaffinity reports {n_cpus} CPUs available to this process")

# _eval_and_grad_list closures take precomputed sin(theta)/cos(theta), not
# theta itself - Propagator.eval_and_grad computes these ONCE per call and
# shares them across every observable (see pprop/propagator/utils.py).
sins0, coss0 = np.sin(params0), np.cos(params0)

single_fn = propagators["sparse"]._eval_and_grad_list[0]
_ = single_fn(sins0, coss0)  # warm-up
N_SINGLE = 2000
t0 = time.perf_counter()
for _ in range(N_SINGLE):
    single_fn(sins0, coss0)
single_call_us = (time.perf_counter() - t0) / N_SINGLE * 1e6
n_obs_total = len(propagators["sparse"]._eval_and_grad_list)

print(f"\nOne observable's eval_and_grad (backend='sparse'): {single_call_us:.1f} us/call")
print(f"  -> {n_obs_total} observables serially: "
      f"~{single_call_us * n_obs_total / 1000:.1f} ms (matches Step 3/10's 'sparse' timing)")
print(
    "  For threading to help, this per-task time needs to dwarf Python's GIL "
    "acquire/release + thread-scheduling overhead per dispatched task - "
    "typically tens of microseconds. A task this size leaves very little room."
)


os.sched_getaffinity reports 128 CPUs available to this process

One observable's eval_and_grad (backend='sparse'): 80.9 us/call
  -> 1000 observables serially: ~80.9 ms (matches Step 3/10's 'sparse' timing)
  For threading to help, this per-task time needs to dwarf Python's GIL acquire/release + thread-scheduling overhead per dispatched task - typically tens of microseconds. A task this size leaves very little room.


In [9]:
def time_threaded(fn_list, params, n_jobs, n_repeat=5):
    """Time evaluating every closure in fn_list at `params`, either serially
    (n_jobs=1) or via a ThreadPoolExecutor of the given width. Each closure
    takes precomputed (sins, coss), recomputed once per `run()` - mirroring
    Propagator.eval_and_grad computing them once per call, not once per
    observable."""
    if n_jobs == 1:
        executor = None
        def run():
            sins, coss = np.sin(params), np.cos(params)
            return [f(sins, coss) for f in fn_list]
    else:
        executor = ThreadPoolExecutor(max_workers=n_jobs)
        def run():
            sins, coss = np.sin(params), np.cos(params)
            return list(executor.map(lambda f: f(sins, coss), fn_list))
    try:
        run()  # warm-up (thread-pool spin-up, cache warming)
        t0 = time.perf_counter()
        for _ in range(n_repeat):
            run()
        return (time.perf_counter() - t0) / n_repeat * 1000
    finally:
        if executor is not None:
            executor.shutdown()


print(f"{'backend':10s}{'eval_n_jobs':>12s}{'ms/call':>10s}{'speedup':>10s}")
sweep_baseline = {}
for backend in ("standard", "sparse"):
    fn_list = propagators[backend]._eval_and_grad_list
    for n_jobs in JOB_COUNTS:
        resolved = n_cpus if n_jobs == -1 else n_jobs
        ms = time_threaded(fn_list, params0, resolved)
        if n_jobs == 1:
            sweep_baseline[backend] = ms
        speedup = sweep_baseline[backend] / ms
        print(f"{backend:10s}{n_jobs:>12d}{ms:10.3f}{speedup:9.2f}x")
        record(backend, resolved, ms, notes="eval_n_jobs sweep")


backend    eval_n_jobs   ms/call   speedup
standard             1   206.301     1.00x
standard             8   220.855     0.93x
standard            16   337.807     0.61x
standard            32   235.543     0.88x
standard            64   237.668     0.87x
standard            -1   246.903     0.84x
sparse               1    81.418     1.00x
sparse               8   116.719     0.70x
sparse              16   175.499     0.46x
sparse              32   128.720     0.63x
sparse              64   126.911     0.64x
sparse              -1   128.924     0.63x


In [10]:
def time_dispatch_overhead(n_tasks, n_jobs, n_repeat=20):
    """Same thread-pool machinery as time_threaded, but every task is a
    no-op - isolates pure dispatch/scheduling overhead from real work."""
    if n_jobs == 1:
        executor = None
        def run():
            return [None for _ in range(n_tasks)]
    else:
        executor = ThreadPoolExecutor(max_workers=n_jobs)
        def run():
            return list(executor.map(lambda _: None, range(n_tasks)))
    try:
        run()
        t0 = time.perf_counter()
        for _ in range(n_repeat):
            run()
        return (time.perf_counter() - t0) / n_repeat * 1000
    finally:
        if executor is not None:
            executor.shutdown()


print(f"\nControl: dispatching {n_obs_total} NO-OP tasks "
      "(isolates thread-pool overhead from any real NumPy work)")
print(f"{'eval_n_jobs':>12s}{'ms (no-op)':>12s}")
for n_jobs in JOB_COUNTS:
    resolved = n_cpus if n_jobs == -1 else n_jobs
    ms = time_dispatch_overhead(n_obs_total, resolved)
    print(f"{n_jobs:>12d}{ms:12.4f}")

print(
    f"\n{n_obs_total} tasks at {single_call_us:.0f} us/task of REAL work = "
    f"~{n_obs_total * single_call_us / 1000:.2f} ms of work to actually parallelise. "
    "Compare that to the no-op dispatch cost above at the same thread counts: "
    "dispatch overhead alone already eats into that budget, before any GIL "
    "contention between threads doing real NumPy work is even considered - "
    "which is why every thread count in the sweep above loses to eval_n_jobs=1."
)



Control: dispatching 1000 NO-OP tasks (isolates thread-pool overhead from any real NumPy work)
 eval_n_jobs  ms (no-op)
           1      0.0184
           8     52.7869
          16     37.0259
          32     34.5266
          64     32.8147
          -1     25.2495

1000 tasks at 81 us/task of REAL work = ~80.94 ms of work to actually parallelise. Compare that to the no-op dispatch cost above at the same thread counts: dispatch overhead alone already eats into that budget, before any GIL contention between threads doing real NumPy work is even considered - which is why every thread count in the sweep above loses to eval_n_jobs=1.


## Step 7: A real sparse-matrix implementation (SciPy)

pprop's `backend="sparse"` is *not* a sparse matrix in the linear-algebra
sense - `idx_sin`/`pow_sin`/`idx_cos`/`pow_cos` are narrow **dense** NumPy
arrays (shape `(n_terms, W)`), just narrower than `"standard"`'s
`(n_terms, num_params)`. The one place an actual sparse *matrix* naturally
fits is the gradient's final scatter-add - `np.add.at(grad, idx_sin.ravel(),
sin_grad_terms.ravel())` routes each term's narrow contribution back into
the shared `(num_params,)` gradient, and "route values into a longer vector
by index, summing collisions" is exactly what a sparse matrix-vector product
does.

**Storage**: identical `idx_sin`/`pow_sin`/`idx_cos`/`pow_cos` gathered
arrays as `"sparse"` (built by the same unmodified `build_sparse_arrays`
from `pprop/propagator/utils.py`), plus one extra artefact per observable: a
`scipy.sparse.csr_matrix` of shape `(num_params, n_terms * W)` encoding
which flattened `(term, w)` slot lands on which parameter. Built once, at
"propagate" time here (mirroring what `Propagator.propagate` does for the
other backends), reused on every call.

**Evaluation**: the same gather + elementwise power/product forward pass as
`"sparse"`. The only change is the last line of the gradient computation:
`S_sin @ sin_grad_terms.ravel()` (a sparse matrix-vector product) instead of
`np.add.at(...)`.


In [11]:
def build_scipy_sparse_evaluator(expr, num_params):
    """Same forward/gradient math as make_sparse_evaluator (pprop's
    "sparse" backend), but the gradient's scatter-add is a real
    scipy.sparse matrix-vector product instead of np.add.at. Takes
    precomputed (sins, coss), same convention as pprop's own closures."""
    coeffs, idx_sin, pow_sin, idx_cos, pow_cos = build_sparse_arrays(expr, num_params)

    def _scatter_matrix(idx):
        rows = idx.ravel()
        cols = np.arange(idx.size)
        data = np.ones(idx.size)
        return sp.csr_matrix((data, (rows, cols)), shape=(num_params, idx.size))

    S_sin = _scatter_matrix(idx_sin)
    S_cos = _scatter_matrix(idx_cos)

    def _eval(sins, coss):
        sin_g, cos_g = sins[idx_sin], coss[idx_cos]
        sin_pow = np.where(pow_sin > 0, sin_g ** pow_sin, 1.0)
        cos_pow = np.where(pow_cos > 0, cos_g ** pow_cos, 1.0)
        return float((coeffs * sin_pow.prod(axis=1) * cos_pow.prod(axis=1)).sum())

    def _eval_grad(sins, coss):
        sin_g, cos_g = sins[idx_sin], coss[idx_cos]
        sin_pow = np.where(pow_sin > 0, sin_g ** pow_sin, 1.0)
        cos_pow = np.where(pow_cos > 0, cos_g ** pow_cos, 1.0)
        sin_prod, cos_prod = sin_pow.prod(axis=1), cos_pow.prod(axis=1)
        term_vals = coeffs * sin_prod * cos_prod

        def excl(pow_arr):
            m = pow_arr.shape[0]
            left = np.cumprod(np.concatenate([np.ones((m, 1)), pow_arr[:, :-1]], axis=1), axis=1)
            right = np.cumprod(np.concatenate([pow_arr[:, 1:], np.ones((m, 1))], axis=1)[:, ::-1], axis=1)[:, ::-1]
            return left * right

        excl_sin, excl_cos = excl(sin_pow), excl(cos_pow)
        cos_at_sin, sin_at_cos = coss[idx_sin], sins[idx_cos]

        d_sin = np.where(pow_sin > 0, pow_sin * np.where(sin_g != 0, sin_g ** (pow_sin - 1), 0.0) * cos_at_sin, 0.0)
        d_cos = np.where(pow_cos > 0, -pow_cos * np.where(cos_g != 0, cos_g ** (pow_cos - 1), 0.0) * sin_at_cos, 0.0)

        sin_grad_terms = coeffs[:, None] * d_sin * excl_sin * cos_prod[:, None]
        cos_grad_terms = coeffs[:, None] * sin_prod[:, None] * excl_cos * d_cos

        grad = S_sin @ sin_grad_terms.ravel() + S_cos @ cos_grad_terms.ravel()  # sparse matmul, not np.add.at
        return float(term_vals.sum()), grad

    return _eval, _eval_grad


scipy_sparse_eval_list, scipy_sparse_eval_and_grad_list = [], []
for expr in exprs:
    f, fg = build_scipy_sparse_evaluator(expr, num_params)
    scipy_sparse_eval_list.append(f)
    scipy_sparse_eval_and_grad_list.append(fg)

vals_scipy = np.array([f(sins0, coss0) for f in scipy_sparse_eval_list])
results_scipy = [f(sins0, coss0) for f in scipy_sparse_eval_and_grad_list]
vals_scipy2 = np.array([v for v, _ in results_scipy])
grads_scipy = np.stack([g for _, g in results_scipy])

assert np.allclose(vals_scipy, reference_vals, atol=1e-8)
assert np.allclose(vals_scipy2, reference_vals, atol=1e-8)
assert np.allclose(grads_scipy, reference_grads, atol=1e-8)
print("scipy.sparse-backed evaluator matches the reference to machine precision.")


scipy.sparse-backed evaluator matches the reference to machine precision.


In [12]:
print(f"{'backend':24s}{'eval_n_jobs':>12s}{'ms/call':>10s}{'speedup':>10s}")
scipy_baseline = None
for n_jobs in JOB_COUNTS:
    resolved = n_cpus if n_jobs == -1 else n_jobs
    ms = time_threaded(scipy_sparse_eval_and_grad_list, params0, resolved)
    if n_jobs == 1:
        scipy_baseline = ms
    speedup = scipy_baseline / ms
    print(f"{'sparse (scipy)':24s}{n_jobs:>12d}{ms:10.3f}{speedup:9.2f}x")
    record("sparse (scipy)", resolved, ms, notes="eval_n_jobs sweep")

print(
    f"\nAt eval_n_jobs=1: sparse (pprop, np.add.at) = {sweep_baseline['sparse']:.3f} ms/call, "
    f"sparse (scipy, sparse matmul) = {scipy_baseline:.3f} ms/call - "
    "the scatter-add is a small fraction of this call's total cost either way "
    "(most of it is the sin/cos/pow/cumprod forward+gradient math), so swapping "
    "np.add.at for a real sparse matrix here barely moves the needle. That's "
    "itself informative: the scatter-add is NOT inherently expensive in plain "
    "NumPy - see Step 9, where the exact same scatter becomes the dominant "
    "cost once it runs inside jax.vmap on CPU instead."
)


backend                  eval_n_jobs   ms/call   speedup
sparse (scipy)                     1    92.511     1.00x
sparse (scipy)                     8   152.769     0.61x
sparse (scipy)                    16   202.137     0.46x
sparse (scipy)                    32   157.460     0.59x
sparse (scipy)                    64   161.664     0.57x
sparse (scipy)                    -1   157.391     0.59x

At eval_n_jobs=1: sparse (pprop, np.add.at) = 81.418 ms/call, sparse (scipy, sparse matmul) = 92.511 ms/call - the scatter-add is a small fraction of this call's total cost either way (most of it is the sin/cos/pow/cumprod forward+gradient math), so swapping np.add.at for a real sparse matrix here barely moves the needle. That's itself informative: the scatter-add is NOT inherently expensive in plain NumPy - see Step 9, where the exact same scatter becomes the dominant cost once it runs inside jax.vmap on CPU instead.


## Step 8: Does JAX support sparse arrays?

Yes - `jax.experimental.sparse` provides `BCOO`/`BCSR`/`COO`/`CSR` sparse
array types that behave like restricted `jax.Array`s: they can be built from
a `scipy.sparse` matrix, multiplied against dense arrays, and (for the
operation that matters here - a constant sparse matrix times a
differentiated dense vector) differentiated through with ordinary
`jax.grad`/`jax.jit`, because the sparse operand isn't what's being
differentiated.

That said, the "experimental" in the module path is accurate, not
boilerplate:

- The API and its numerics are explicitly unstable across JAX releases.
- Most sparse-sparse operations (as opposed to sparse-dense) have limited or
  no support, and there's no dense-array-style broadcasting.
- A `BCOO` array's number of stored elements (`nse`) is a fixed, static
  shape - like everything else in JAX, a sparsity pattern that changes
  between calls forces a recompile, same as any other shape change.

The quick check below just confirms the basics work in this environment
before Step 9 uses it for something real: swapping in a `BCOO` scatter for
the `.at[idx].add()` gradient scatter that
`tests/test_eval_and_grad_jit_bench.py` identified as `"vmap"`'s CPU
bottleneck (~20-24x the forward pass's cost there).


In [13]:
import jax
import jax.numpy as jnp
import jax.experimental.sparse as jsparse

print(f"jax version: {jax.__version__}")

# Minimal correctness/grad check: a small sparse matrix times a dense vector,
# differentiated w.r.t. the dense vector.
demo_scipy = sp.random(20, 30, density=0.1, random_state=0, format="csr")
demo_bcoo = jsparse.BCOO.from_scipy_sparse(demo_scipy)
print(f"BCOO from a 20x30 scipy matrix: shape={demo_bcoo.shape}, nse={demo_bcoo.nse}")


def demo_loss(v):
    return jnp.sum((demo_bcoo @ v) ** 2)


v0 = jnp.ones(30)
val, grad = jax.value_and_grad(demo_loss)(v0)
print(f"value={float(val):.4f}, grad.shape={grad.shape}  (grad w.r.t. the DENSE operand works fine)")


jax version: 0.5.0
BCOO from a 20x30 scipy matrix: shape=(20, 30), nse=60
value=69.5618, grad.shape=(30,)  (grad w.r.t. the DENSE operand works fine)


## Step 9: Trying a JAX sparse scatter on `"vmap"`'s bottleneck

`tests/test_eval_and_grad_jit_bench.py` (section 5b) isolated exactly where
`backend="vmap"` loses time on this CPU-only box: not the forward pass
(competitive with NumPy), but the gradient's final scatter-add -
`jnp.zeros(num_params).at[idx].add(...)` - which measured ~20-24x the cost
of the forward pass alone. Step 8 showed `jax.experimental.sparse` can do a
sparse matrix-vector product with ordinary autodiff support, and Step 7
found that a literal sparse-matrix scatter costs about the same as
`np.add.at` in plain NumPy. The natural question: does replacing
`.at[idx].add()` with a `BCOO` sparse matmul fix `"vmap"`'s scatter
bottleneck the same way?

**Storage**: the same padded `(n_obs, max_terms, W)` gathered arrays as
`"vmap"` (Step 2), plus one big block-diagonal sparse matrix - shape
`(n_obs * num_params, n_obs * max_terms * W)` - built once with
`scipy.sparse` and converted to a `jax.experimental.sparse.BCOO`. Each
observable only ever scatters into its own `num_params`-wide slice of the
output, so the matrix is block-diagonal across observables - this just
batches Step 7's per-observable scatter matrix into one call instead of one
Python-level matrix per observable.

**Evaluation**: the same `jax.vmap`+`jax.jit`-batched forward pass and
per-term local gradient contributions as `"vmap"`, stopping short of the
final scatter. Then one sparse matrix-vector product (`S @
grad_terms.ravel()`) replaces the batched `.at[idx].add()` call, and the
flat result is reshaped back to `(n_obs, num_params)`.

**A real compile-time trap, and its fix:** the first version of this cell
built `S_sin`/`S_cos` once and closed over their `.data`/`.indices` as plain
Python constants, the same way `"vmap"`'s own `arrs` dict is closed over in
Step 2. At the full 1000-observable scale that made compilation
pathological - `slow_operation_alarm` messages kept appearing with growing
durations (1s, then 23s, 24s, 26s, 28s, 49s, ... - still climbing after
several minutes, never finished). The alarms named `compare`/`slice` ops
over million-element arrays matching this batch's `nse` - XLA was trying to
*eagerly constant-fold* `BCOO`'s own internal index-validity checks, since
every input to those ops was a compile-time-known Python constant (this
doesn't happen to `"vmap"`'s plain `.at[idx].add()`, which has no such
validation step to fold). Passing those SAME `data`/`indices` arrays as
explicit *arguments* to the jitted function instead - shown below - fixes
this completely: they're runtime values now, not literals, so XLA has
nothing to eagerly fold; the checks just compile into fast, ordinary
vectorised ops. That drops compile time from "several minutes, not yet
finished" to about a second, so the cell below runs on the real full
1000-observable batch, not a stand-in subsample.


In [14]:
def build_grad_terms_batched(exprs, num_params):
    """Same forward pass + local per-term gradient contributions as
    vmap_backend._single_eval, but stopping BEFORE the final scatter into
    (num_params,) - that's done separately below by a sparse matmul."""
    per_obs = [build_sparse_arrays(e, num_params) for e in exprs]
    n_obs = len(per_obs)
    max_terms = max(c.shape[0] for c, *_ in per_obs) or 1
    max_ws = max(idx_sin.shape[1] for _, idx_sin, *_ in per_obs) or 1
    max_wc = max(idx_cos.shape[1] for *_, idx_cos, _ in per_obs) or 1

    coeffs_b = np.zeros((n_obs, max_terms))
    idx_sin_b = np.zeros((n_obs, max_terms, max_ws), dtype=np.int64)
    pow_sin_b = np.zeros((n_obs, max_terms, max_ws))
    idx_cos_b = np.zeros((n_obs, max_terms, max_wc), dtype=np.int64)
    pow_cos_b = np.zeros((n_obs, max_terms, max_wc))
    for i, (coeffs, idx_sin, pow_sin, idx_cos, pow_cos) in enumerate(per_obs):
        n = coeffs.shape[0]
        coeffs_b[i, :n] = coeffs
        idx_sin_b[i, :n, : idx_sin.shape[1]] = idx_sin
        pow_sin_b[i, :n, : pow_sin.shape[1]] = pow_sin
        idx_cos_b[i, :n, : idx_cos.shape[1]] = idx_cos
        pow_cos_b[i, :n, : pow_cos.shape[1]] = pow_cos

    def _val_and_local_grads(thetas, coeffs, idx_sin, pow_sin, idx_cos, pow_cos):
        sin_g, cos_g = jnp.sin(thetas)[idx_sin], jnp.cos(thetas)[idx_cos]
        sin_pow = jnp.where(pow_sin > 0, sin_g ** pow_sin, 1.0)
        cos_pow = jnp.where(pow_cos > 0, cos_g ** pow_cos, 1.0)
        sin_prod, cos_prod = jnp.prod(sin_pow, axis=-1), jnp.prod(cos_pow, axis=-1)
        val = jnp.sum(coeffs * sin_prod * cos_prod)

        def excl(p):
            n = p.shape[0]
            left = jnp.cumprod(jnp.concatenate([jnp.ones((n, 1)), p[:, :-1]], axis=1), axis=1)
            right = jnp.cumprod(jnp.concatenate([p[:, 1:], jnp.ones((n, 1))], axis=1)[:, ::-1], axis=1)[:, ::-1]
            return left * right

        excl_sin, excl_cos = excl(sin_pow), excl(cos_pow)
        d_sin = jnp.where(pow_sin > 0, pow_sin * jnp.where(sin_g != 0, sin_g ** (pow_sin - 1), 0.0) * jnp.cos(thetas)[idx_sin], 0.0)
        d_cos = jnp.where(pow_cos > 0, -pow_cos * jnp.where(cos_g != 0, cos_g ** (pow_cos - 1), 0.0) * jnp.sin(thetas)[idx_cos], 0.0)

        sin_grad_terms = coeffs[:, None] * d_sin * excl_sin * cos_prod[:, None]
        cos_grad_terms = coeffs[:, None] * sin_prod[:, None] * excl_cos * d_cos
        return val, sin_grad_terms, cos_grad_terms

    batched = jax.jit(jax.vmap(_val_and_local_grads, in_axes=(None, 0, 0, 0, 0, 0)))
    arrays = dict(coeffs=jnp.asarray(coeffs_b), idx_sin=jnp.asarray(idx_sin_b), pow_sin=jnp.asarray(pow_sin_b),
                  idx_cos=jnp.asarray(idx_cos_b), pow_cos=jnp.asarray(pow_cos_b))
    return batched, arrays, idx_sin_b, idx_cos_b, n_obs, max_ws, max_wc


def build_block_scatter(idx_b, n_obs, num_params, W):
    """Block-diagonal sparse matrix: flattened per-observable local (term, w)
    contributions -> that observable's own (num_params,) gradient slice."""
    max_terms = idx_b.shape[1]
    obs_offset = np.repeat(np.arange(n_obs) * num_params, max_terms * W)
    rows = obs_offset + idx_b.reshape(-1)
    cols = np.arange(idx_b.size)
    data = np.ones(idx_b.size)
    return sp.csr_matrix((data, (rows, cols)), shape=(n_obs * num_params, idx_b.size))


t0 = time.perf_counter()
batched_val_localgrad, arrs, idx_sin_b, idx_cos_b, n_obs, max_ws, max_wc = build_grad_terms_batched(
    exprs, num_params  # the FULL 1000-observable batch, not a subsample
)
S_sin = jsparse.BCOO.from_scipy_sparse(build_block_scatter(idx_sin_b, n_obs, num_params, max_ws))
S_cos = jsparse.BCOO.from_scipy_sparse(build_block_scatter(idx_cos_b, n_obs, num_params, max_wc))
S_sin_shape, S_cos_shape = S_sin.shape, S_cos.shape  # plain python tuples - fine to close over,
                                                      # it's only large ARRAY data that must be
                                                      # passed as an argument, not shape metadata
build_s = time.perf_counter() - t0
print(f"Built batched arrays + BCOO scatter matrices ({S_sin.nse + S_cos.nse:,} nonzeros total, "
      f"{n_obs} observables) in {build_s:.2f}s")


@jax.jit
def eval_and_grad_jax_sparse_scatter(thetas, coeffs, idx_sin, pow_sin, idx_cos, pow_cos,
                                      S_sin_data, S_sin_idx, S_cos_data, S_cos_idx):
    """S_sin/S_cos's data/indices are passed as ARGUMENTS here, not closed
    over - see the markdown above for why that's what makes this compile in
    ~1s instead of pathologically slow at this scale."""
    vals, sin_terms, cos_terms = batched_val_localgrad(thetas, coeffs, idx_sin, pow_sin, idx_cos, pow_cos)
    S_sin_l = jsparse.BCOO((S_sin_data, S_sin_idx), shape=S_sin_shape)
    S_cos_l = jsparse.BCOO((S_cos_data, S_cos_idx), shape=S_cos_shape)
    grad_flat = S_sin_l @ sin_terms.reshape(-1) + S_cos_l @ cos_terms.reshape(-1)
    return vals, grad_flat.reshape(n_obs, num_params)


t0 = time.perf_counter()
vals_jsp, grads_jsp = eval_and_grad_jax_sparse_scatter(
    jnp.asarray(params0), arrs["coeffs"], arrs["idx_sin"], arrs["pow_sin"], arrs["idx_cos"], arrs["pow_cos"],
    S_sin.data, S_sin.indices, S_cos.data, S_cos.indices,
)
jax.block_until_ready((vals_jsp, grads_jsp))
compile_s = time.perf_counter() - t0
print(f"First call (trace + XLA compile) on the FULL {n_obs}-observable batch: {compile_s:.2f}s")

vals_jsp, grads_jsp = np.asarray(vals_jsp), np.asarray(grads_jsp)
assert np.allclose(vals_jsp, reference_vals, atol=1e-6)
assert np.allclose(grads_jsp, reference_grads, atol=1e-6)
print("jax sparse-scatter evaluator matches the reference (full batch).")


Built batched arrays + BCOO scatter matrices (20,385,000 nonzeros total, 1000 observables) in 0.54s
First call (trace + XLA compile) on the FULL 1000-observable batch: 0.80s
jax sparse-scatter evaluator matches the reference (full batch).


In [15]:
N_JSP = 5
t0 = time.perf_counter()
for _ in range(N_JSP):
    vals_jsp, grads_jsp = eval_and_grad_jax_sparse_scatter(
        jnp.asarray(params0), arrs["coeffs"], arrs["idx_sin"], arrs["pow_sin"], arrs["idx_cos"], arrs["pow_cos"],
        S_sin.data, S_sin.indices, S_cos.data, S_cos.indices,
    )
jax.block_until_ready((vals_jsp, grads_jsp))
jax_sparse_ms = (time.perf_counter() - t0) / N_JSP * 1000
print(f"Steady state (jax sparse BCOO scatter, all {n_obs} observables): {jax_sparse_ms:.3f} ms/call")

record("vmap + jax sparse scatter (BCOO)", None, jax_sparse_ms,
       notes=f"full {n_obs}-obs batch; one-time compile: {compile_s:.2f}s (args, not closures - see markdown)")

vmap_full_ms = next(r["ms_per_call"] for r in ALL_RESULTS if r["implementation"] == "vmap")
sparse_full_ms = sweep_baseline["sparse"]

print(
    f"\nResult: at the FULL {n_obs}-observable scale, a jax.experimental.sparse BCOO scatter "
    f"measures {jax_sparse_ms:.1f} ms/call vs. plain vmap's .at[idx].add() at {vmap_full_ms:.1f} "
    f"ms/call (Step 3) - a real {vmap_full_ms / jax_sparse_ms:.2f}x speedup within the vmap family, "
    f"with a one-time compile of just {compile_s:.2f}s once the compile-time trap above is avoided. "
    "So: yes, JAX supports sparse arrays (Step 8), and here they DO help - the scatter-add "
    "bottleneck tests/test_eval_and_grad_jit_bench.py identified for \"vmap\" is real, and a "
    "properly-wired BCOO scatter measurably closes part of the gap. It doesn't close all of it "
    f"though: plain NumPy's 'sparse' backend is still faster ({sparse_full_ms:.1f} ms/call) - its "
    "narrow gather-based forward pass wins on this CPU-only box regardless of how the gradient's "
    "scatter is implemented, so the Step 10 recommendation doesn't change. The generalisable lesson "
    "is more about JAX than about this specific backend: when a jitted function's compile time "
    "blows up unexpectedly, check whether large arrays are being closed over as Python constants "
    "rather than passed as traced arguments before concluding the underlying computation doesn't "
    "scale - XLA's eager constant-folding can eagerly \"evaluate\" a purely-constant graph at "
    "compile time, and that evaluation is not the same fast code path as running it at runtime."
)


Steady state (jax sparse BCOO scatter, all 1000 observables): 19.648 ms/call

Result: at the FULL 1000-observable scale, a jax.experimental.sparse BCOO scatter measures 19.6 ms/call vs. plain vmap's .at[idx].add() at 20.6 ms/call (Step 3) - a real 1.05x speedup within the vmap family, with a one-time compile of just 0.80s once the compile-time trap above is avoided. So: yes, JAX supports sparse arrays (Step 8), and here they DO help - the scatter-add bottleneck tests/test_eval_and_grad_jit_bench.py identified for "vmap" is real, and a properly-wired BCOO scatter measurably closes part of the gap. It doesn't close all of it though: plain NumPy's 'sparse' backend is still faster (81.4 ms/call) - its narrow gather-based forward pass wins on this CPU-only box regardless of how the gradient's scatter is implemented, so the Step 10 recommendation doesn't change. The generalisable lesson is more about JAX than about this specific backend: when a jitted function's compile time blows up unexpec

## Step 10: Summary

Every measurement taken in this notebook, printed as one plain-Python table
(deliberately not markdown - re-running the notebook regenerates this
exactly, rather than someone hand-copying numbers into a table that then
drifts out of sync with the code).


In [16]:
serial_baseline = min(
    r["ms_per_call"] for r in ALL_RESULTS
    if r["implementation"] == "sparse" and r["eval_n_jobs"] == 1
)

header = f"{'implementation':32s}{'eval_n_jobs':>12s}{'ms/call':>10s}{'speedup':>9s}  notes"
print("=" * 78)
print("Full results, fastest first "
      f"(speedup relative to backend='sparse', eval_n_jobs=1: {serial_baseline:.3f} ms/call)")
print("=" * 78)
print(header)
print("-" * len(header))

for r in sorted(ALL_RESULTS, key=lambda r: r["ms_per_call"]):
    jobs_str = "-" if r["eval_n_jobs"] is None else str(r["eval_n_jobs"])
    speedup = serial_baseline / r["ms_per_call"]
    print(f"{r['implementation']:32s}{jobs_str:>12s}{r['ms_per_call']:10.3f}"
          f"{speedup:8.2f}x  {r['notes']}")

print("-" * len(header))


Full results, fastest first (speedup relative to backend='sparse', eval_n_jobs=1: 81.418 ms/call)
implementation                   eval_n_jobs   ms/call  speedup  notes
----------------------------------------------------------------------
vmap + jax sparse scatter (BCOO)           -    19.648    4.14x  full 1000-obs batch; one-time compile: 0.80s (args, not closures - see markdown)
vmap                                       -    20.638    3.95x  single-threaded baseline
sparse                                     1    81.418    1.00x  eval_n_jobs sweep
sparse (scipy)                             1    92.511    0.88x  eval_n_jobs sweep
sparse                                     8   116.719    0.70x  eval_n_jobs sweep
sparse                                    64   126.911    0.64x  eval_n_jobs sweep
sparse                                    32   128.720    0.63x  eval_n_jobs sweep
sparse                                   128   128.924    0.63x  eval_n_jobs sweep
sparse (scipy)            

### Conclusions

- **`eval_n_jobs` doesn't help, at any thread count, for either
  `"standard"` or `"sparse"`** (Step 6) - not because 128 cores aren't
  there, but because a single observable's `eval_and_grad` call is on the
  order of 0.1-0.2 ms, too small for NumPy's released-GIL work to amortise
  Python's per-task GIL/scheduling overhead. The no-op dispatch control in
  Step 6 shows that overhead directly, independent of what the task does.
- **A literal `scipy.sparse` scatter (Step 7) ties `np.add.at` almost
  exactly** - confirming the scatter-add itself isn't expensive in plain
  NumPy at this problem's scale; `"sparse"`'s edge over `"standard"` (see
  `tests/test_eval_and_grad_jit_bench.py`) comes from the narrower
  *forward* arrays, not the gradient scatter.
- **JAX does support sparse arrays** (Step 8, `jax.experimental.sparse`),
  and swapping one in for `"vmap"`'s `.at[idx].add()` scatter (Step 9) is a
  genuine, measurable win *within the vmap family* - once a real
  compile-time trap is avoided (large arrays must be passed as jit
  *arguments*, not closed-over Python constants, or XLA eagerly
  constant-folds `BCOO`'s internal validation checks at compile time,
  which can blow up to several minutes for million-element arrays). Even
  fixed, `"vmap"` + `BCOO` still trails `"sparse"` - NumPy's narrower
  gather-based forward pass wins here regardless of the scatter method.
- **Recommendation unchanged**: `backend="sparse"`, `eval_n_jobs=1`. Every
  parallelisation lever tried in this notebook that could plausibly beat it
  - thread pools at any width, a real sparse matrix for the scatter, JAX's
  own sparse arrays - was either neutral, a regression, or (for JAX sparse)
  a genuine improvement that still didn't close the gap, at this workload's
  scale (tens to a few hundred terms per observable, ~1000 observables).
  `num_jobs` at *propagation* time (separate processes, no GIL) and GPU
  hardware remain the two untested levers that could plausibly change this
  - see `Propagator.propagate`'s docstring and
  `tests/test_eval_and_grad_jit_bench.py` for what's already been tried and
  what hasn't.

See also:
- [`Propagator.propagate`](../src/pprop/propagator/__init__.py)'s docstring - the
  authoritative reference for every parameter mentioned here.
- [`tests/test_backends.py`](../tests/test_backends.py) - the correctness
  test the three built-in backends are held to.
- [`tests/test_eval_and_grad_jit_bench.py`](../tests/test_eval_and_grad_jit_bench.py) -
  the original performance investigation this notebook builds on, including
  the `"vmap"` scatter-add finding Step 9 partially fixes with a real
  `jax.experimental.sparse` scatter.
